<a href="https://colab.research.google.com/github/sejal-godbole/Federated-Learning/blob/main/FL_Assignment5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import Libraries and Define the Model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy

In [ ]:
# 1. Define a very simple Neural Network
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc = nn.Linear(10, 2) # 10 input features, 2 output classes

    def forward(self, x):
        return self.fc(x)

In [ ]:
# Create the Global Model (This lives on the "Server")
global_model = SimpleNet()
print("Global model created!")

Global model created!


Create a Local Client Training Function

In [ ]:
def client_train(client_id, model, data_x, data_y, epochs=3, lr=0.01):
    """Simulates local training on a client device."""
    print(f"--- Client {client_id} starting local training ---")

    # Use a standard loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    # Train for a few epochs
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(data_x)
        loss = criterion(outputs, data_y)
        loss.backward()
        optimizer.step()

    print(f"Client {client_id} finished. Final Loss: {loss.item():.4f}")

    # Return the updated weights (state_dict) AND the number of data samples
    num_samples = len(data_x)
    return model.state_dict(), num_samples

Implement Server-Side Aggregation (Weighted FedAvg)

In [ ]:
def server_aggregate(global_model, client_updates):
    """Aggregates client weights using Weighted Federated Averaging."""

    # 1. Calculate total number of data samples across all clients (the 'N' in our formula)
    total_samples = sum([num_samples for _, num_samples in client_updates])

    # 2. Get the current weights of the global model
    global_weights = global_model.state_dict()

    # 3. Loop through every layer/weight in the model
    for key in global_weights.keys():
        # Start with zeros for the new weights
        global_weights[key] = torch.zeros_like(global_weights[key])

        # Add the weighted contribution from each client
        for client_weights, num_samples in client_updates:
            # Weight formula: client_weight * (n_k / N)
            weight_ratio = num_samples / total_samples
            global_weights[key] += client_weights[key] * weight_ratio

    # 4. Load the newly aggregated weights back into the global model
    global_model.load_state_dict(global_weights)
    print("\n[Server] Global model updated with Weighted FedAvg!\n")

The Federated Learning Loop

In [ ]:
# Create fake data for Client A (has 100 samples)
client_A_x = torch.randn(100, 10)
client_A_y = torch.randint(0, 2, (100,))

# Create fake data for Client B (has only 20 samples)
client_B_x = torch.randn(20, 10)
client_B_y = torch.randint(0, 2, (20,))

# Total Communication Rounds between Server and Clients
num_rounds = 3

for round_num in range(1, num_rounds + 1):
    print(f"=== Communication Round {round_num} ===")

    # 1. Global Model Distribution: Copy global model for each client
    # (We copy so they don't overwrite the original model in memory)
    model_A = copy.deepcopy(global_model)
    model_B = copy.deepcopy(global_model)

    # 2. Local Training: Clients train on their own data
    weights_A, samples_A = client_train("A", model_A, client_A_x, client_A_y)
    weights_B, samples_B = client_train("B", model_B, client_B_x, client_B_y)

    # Collect updates
    client_updates = [(weights_A, samples_A), (weights_B, samples_B)]

    # 3. Server-Side Aggregation
    server_aggregate(global_model, client_updates)

print("Federated Learning process complete!")

=== Communication Round 1 ===
--- Client A starting local training ---
Client A finished. Final Loss: 0.7511
--- Client B starting local training ---
Client B finished. Final Loss: 0.7629

[Server] Global model updated with Weighted FedAvg!

=== Communication Round 2 ===
--- Client A starting local training ---
Client A finished. Final Loss: 0.7481
--- Client B starting local training ---
Client B finished. Final Loss: 0.7576

[Server] Global model updated with Weighted FedAvg!

=== Communication Round 3 ===
--- Client A starting local training ---
Client A finished. Final Loss: 0.7452
--- Client B starting local training ---
Client B finished. Final Loss: 0.7524

[Server] Global model updated with Weighted FedAvg!

Federated Learning process complete!
